In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import datetime as dt

from src.data_preparation.input_preparation import prepare_input

In [4]:
df = pd.read_csv("ETT-small/ETTh1.csv")
df_prepped = prepare_input(df)

df = df_prepped

In [5]:
test_period_cutoff = dt.datetime(2017, 9, 1, 0, 0, 0)

df_train = df[df['date'] < test_period_cutoff]
df_test = df[df['date'] >= test_period_cutoff]

X_train = df_train.drop(columns=['date', 'OT'])
y_train = df_train['OT']

X_test = df_test.drop(columns=['date', 'OT'])
y_test = df_test['OT']

## Arima

In [21]:
from src.model.arima import fit_arima, predict_arima
arima_model = fit_arima(y_train, X_train)
# arima_model.summary()

In [30]:
y_pred = predict_arima(arima_model, y_test, X_test)
SSE = ((y_pred - y_test) ** 2).sum()
WMAPE = (abs(y_pred - y_test).sum() / y_test.sum()) * 100
print(f"Arima sum of Squared Errors (SSE): {SSE}")
print(f"Arima Weighted Mean Absolute Percentage Error (WMAPE): {WMAPE}%")

Arima sum of Squared Errors (SSE): 35995.1194423994
Arima Weighted Mean Absolute Percentage Error (WMAPE): 22.470179613798994%


## LightGBM

In [10]:
from src.model.lightGBM import make_features
df_feat, feature_cols = make_features(df_train, y_col='OT', X_cols=X_train.columns.tolist())

y_tr = df_feat.dropna()['OT']
df_feat = df_feat.dropna(subset=feature_cols)  # lose first 168 rows to rolling window

X_tr = df_feat[feature_cols]

In [11]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr)],  # ideally use a proper time-based validation split, see note below
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)

/home/robert-slob/Documents/GitHub/ETDataset/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/robert-slob/Documents/GitHub/ETDataset/.venv/lib/python3.12/site-packages/lightgbm/callback.py:347: UserWarning: Only training set found, disabling early stopping.
  _log_warning("Only training set found, disabling early stopping.")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000383 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 10080, number of used features: 19
[LightGBM] [Info] Start training from score 17.217577
[50]	training's l2: 1.15863
[100]	training's l2: 0.426159
[150]	training's l2: 0.288031
[200]	training's l2: 0.213415
[250]	training's l2: 0.167575
[300]	training's l2: 0.132566
[350]	training's l2: 0.108751
[400]	training's l2: 0.0894608
[450]	training's l2: 0.0750841
[500]	training's l2: 0.0632773


,learning_rate,0.05
,n_estimators,500
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [31]:
import numpy as np

def rolling_forecast_lgb(model, history_df, X_test, y_test, y_col, feature_cols, lags, rolling_windows, horizon=24):
    history = history_df.copy()
    all_preds = []

    for day_start in range(0, len(X_test), horizon):
        X_day = X_test.iloc[day_start:day_start+horizon]
        y_day = y_test.iloc[day_start:day_start+horizon]

        day_preds = []
        day_history = history.copy()

        # within-day: still recursive, since you only have real y up to the start of the day
        for i in range(len(X_day)):
            next_row = X_day.iloc[[i]].copy()
            next_row[y_col] = np.nan
            day_history = pd.concat([day_history, next_row])

            feat_df, _ = make_features(day_history, y_col=y_col, X_cols=X_day.columns,
                                         lags=lags, rolling_windows=rolling_windows)
            x_input = feat_df.iloc[[-1]][feature_cols]
            y_hat = model.predict(x_input)[0]

            day_preds.append(y_hat)
            day_history.iloc[-1, day_history.columns.get_loc(y_col)] = y_hat

        all_preds.extend(day_preds)

        # NOW feed the real y_day into the master history before next day
        real_day = X_day.copy()
        real_day[y_col] = y_day.values
        history = pd.concat([history, real_day])

    return pd.Series(all_preds, index=X_test.index)

y_pred_lgb_rolling = rolling_forecast_lgb(
    lgb_model, df_train, df_test, y_test, y_col='OT',
    feature_cols=feature_cols, lags=(1,2,24,25,48), rolling_windows=(24,168),
    horizon=24,
)

In [33]:
y_pred_lgb_rolling
SSE = ((y_pred_lgb_rolling - y_test) ** 2).sum()
WMAPE = (abs(y_pred_lgb_rolling - y_test).sum() / y_test.sum()) * 100
print(f"Arima sum of Squared Errors (SSE): {SSE}")
print(f"Arima Weighted Mean Absolute Percentage Error (WMAPE): {WMAPE}%")

Arima sum of Squared Errors (SSE): 34763.0829147501
Arima Weighted Mean Absolute Percentage Error (WMAPE): 21.741729569444356%
